In [3]:
import os
from dotenv import load_dotenv
from groq import Groq
import json

# Load environment variables from .env file
load_dotenv(override=True)

# Get API key
api_key = os.getenv('GROQ_API_KEY')
if not api_key:
    raise ValueError("GROQ_API_KEY not found in .env file")

# Initialize Groq client
client = Groq(api_key=api_key)

In [4]:
# Generate synthetic financial transactions
prompt = """Generate exactly 25 realistic financial transactions in valid JSON format.
Each transaction must have these fields:
- transaction_id: unique identifier (e.g., "TXN001")
- amount: transaction amount in MYR (e.g., 150.75)
- date: ISO date format (e.g., "2026-03-15")
- category: transaction category (e.g., "groceries", "electronics", "utilities")
- merchant: merchant name (e.g., "Speedmart", "Amazon", "Lazada")

Return ONLY the JSON array, no other text.
Example format: [{"transaction_id": "TXN001", "amount": 45.99, "date": "2026-03-15", "category": "groceries", "merchant": "Lazada"}, ...]"""

print("Generating synthetic transactions from Groq...\n")

# Stream the response
completion = client.chat.completions.create(
    model="qwen/qwen3-32b",
    messages=[
        {
            "role": "user",
            "content": prompt
        }
    ],
    temperature=0.6,
    max_completion_tokens=4096,
    top_p=0.95,
    stream=True,
)

# Collect streamed response
response_text = ""
for chunk in completion:
    content = chunk.choices[0].delta.content or ""
    response_text += content
    print(content, end="", flush=True)

print("\n\nParsing transactions...")

# Parse JSON response
try:
    # Extract JSON from response (remove <think>...</think> tags if present)
    import re
    json_match = re.search(r'\[.*\]', response_text, re.DOTALL)
    if json_match:
        transactions = json.loads(json_match.group(0))
    else:
        raise ValueError("No JSON array found in response")
    
    print(f"\n✓ Successfully generated {len(transactions)} transactions\n")
    
    # Display formatted output
    print(json.dumps(transactions, indent=2))
    
    # Optional: Save to file
    with open('generated_transactions.json', 'w') as f:
        json.dump(transactions, f, indent=2)
    print(f"\n✓ Transactions saved to 'generated_transactions.json'")
    
except (json.JSONDecodeError, ValueError) as e:
    print(f"✗ Error parsing JSON: {e}")

Generating synthetic transactions from Groq...

<think>
Okay, I need to generate exactly 25 realistic financial transactions in valid JSON format. Let me start by understanding the requirements. Each transaction must have a unique transaction ID, an amount in MYR, a date in ISO format, a category, and a merchant. The example provided uses formats like "TXN001" for the ID, so I'll follow that pattern, incrementing the number each time.

First, I'll list out possible categories. The example includes groceries, electronics, utilities. I should add more like transportation, dining, entertainment, health, clothing, etc. For merchants, common ones in Malaysia would be Speedmart, Tesco, Digi, Shell, McDonald's, Netflix, etc. I need to ensure variety to look realistic.

Next, the dates should be spread across a reasonable time frame. Let's pick dates from 2026-01-01 to 2026-03-31. I'll randomize the days but keep the months varied. For amounts, they should be realistic for each category. Groce